# Map SPECIATE species to CRACMM

---
    author: Nash Skipper
    date: 2024-08-09

    updated: Michael Pye
    date: 2025-02-27
    
    updated for CRACMM3: Havala Pye
    date: 2025-04-17
---

## Notebook Description
This notebook identifies the CRACMM species for each SPECIATE species using the mapper. The cracmm_mapper function depends on [rdkit](https://www.rdkit.org/).

## Download Notebook
Click [here](https://github.com/USEPA/CRACMM/blob/main/utilities/SPECIATE_map2cracmm.ipynb) to access the Jupyter Notebook file directly in GitHub where it can be downloaded.  

## Setup

In [1]:
import numpy as np
import pandas as pd
import warnings
import os

In [2]:
## Install rdkit if not already installed

# !python -m pip install --user rdkit

# to install in the current kernel:
# %pip install rdkit

In [3]:
# set location of mapper downloaded from https://github.com/USEPA/CRACMM/
# import sys
# utildir = '/path/to/cracmm/utilities/directory'   
# sys.path.append(utildir)

# Import the python utilities
import cracmm1_mapper as cracmm1   # includes: get_cracmm_roc(smiles,koh,log10cstar) (Version 1)
import cracmm2_mapper as cracmm2   # includes: get_cracmm_roc(smiles,koh,log10cstar) (Version 2)
import cracmm3_mapper as cracmm3   # includes: get_cracmm_roc(smiles,koh,log10cstar) (Version 3 alpha)

In [4]:
datadir = '../emissions/SPECIATEInputs/'    # data files of mappings

outputdir = os.path.join(os.getcwd(), 'output/')

In [5]:
pd.set_option('display.max_rows', None)
pd.options.mode.copy_on_write = True
warnings.simplefilter('ignore') # ignore warnings (comment out to see warnings for species that could not be mapped)
csvout_kw = dict(sep=',', na_rep='', float_format=None, columns=None, header=True, index=False)

## SPECIATE Mapping

[Input SPECIATE file](https://github.com/USEPA/CRACMM/tree/main/emissions/SPECIATEInputs)

In [6]:
filename = datadir + 'SPECIATEv5.2x_fromCRACMM.csv' 
df = pd.read_csv(filename)
# for checking if any species mapping changed
orig_map_colname = 'CRACMM2'    # select mechanism version to compare new mapping to

### Calculate C* from Vapor Pressure

$$ C^* \text{must have units of } {\mu g \over m^3} $$
$$ C^* = {p * M * 10^6 \over R * T} $$
$$ \text{p and M are from the input csv file} $$
$$ p = \text{vapor pressure} \left( Pa \right) $$
$$ M = \text{molecular weight} \left( g \over mol\right ) $$
$$ R = 8.314 \text{ } {m^3 * Pa \over mol * K} $$
$$ T = 298 \text{ } K $$

In [7]:
vp_k = 'VP_Pascal_OPERA'
mw_k = 'SPEC_MW'
R = 8.314
T = 298
df['log10Cstar_ugm3'] = np.log10(df[vp_k] * df[mw_k]*10**6 / (R * T))

### Run CRACMM3 Mapper and see mapping changes from CRACMM2 to CRACMM3

In [8]:
smiles_k = 'Smiles Notation'
koh_k    = 'ATMOSPHERIC_HYDROXYLATION_RATE_(AOH)_CM3/MOLECULE*SEC_OPERA'
cstar_k  = 'log10Cstar_ugm3'
df['CRACMM3'] = df.apply(lambda x: cracmm3.get_cracmm_roc(x[smiles_k], x[koh_k], x[cstar_k]), axis=1)

# check if any species mappings changed
df_checkmatch = df.eval(f'match = {orig_map_colname}==CRACMM3')
display_changes = True
show_cols = ['SPECIES_NAME','CRACMM2','CRACMM3']
if len(df_checkmatch[df_checkmatch.match==False])>0:
    print(f'there are {len(df_checkmatch[df_checkmatch.match==False])} changes')
    print(f'out of {len(df_checkmatch)} total species')
    print('the species mappings below changed from CRACMM2')
    display(df_checkmatch[show_cols][df_checkmatch.match==False])
else:
    print('all species matched CRACMM2 mapping')

# save output
df = df.drop(columns=[orig_map_colname, cstar_k])
#df.to_csv(outputdir+'SPECIATEv5.2x_fromCRACMM.csv', **csvout_kw)

there are 239 changes
out of 2890 total species
the species mappings below changed from CRACMM2


,SPECIES_NAME,CRACMM2,CRACMM3
94,1-Methylnaphthalene,NAPH,VROCP6ARO
132,"2,4,5-trichlorophenol",PHEN,CSL
184,2-methylnaphthalene,NAPH,VROCP6ARO
491,Methylnaphthalenes,NAPH,VROCP6ARO
525,Naphthalene,NAPH,VROCP6ARO
561,Phenol (or carbolic acid),PHEN,CSL
680,Acenaphthene,NAPH,VROCP5ARO
681,Acenaphthylene,NAPH,VROCP5ARO
686,Anthracene,NAPH,VROCP2ALK
688,Benz(a)anthracene,NAPH,VROCP0ALK
